# Security-Constrained Unit Commitment (SCUC) - Example 1
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

> **Model overview:** Two-generator, two-period MILP SCUC.
> Variables: commitment `u[g,t]` (binary), startup indicator `v[g,t]` (binary), dispatch `Pg[g,t]` (continuous ≥ 0).
> Costs: operating cost + no-load cost + startup cost.
> Constraints: power balance, generation limits, ramp-up/down with Big-M relaxation at startup/shutdown, startup logic.

In [2]:
from pyomo.environ import *
import time

# ─────────────────────────────────────────────────────────────────
# System Data
# ─────────────────────────────────────────────────────────────────
BigM = 1e3

# Generators: {g: {min MW, max MW, RR MW/period, OpCost $/MWh, NlCost $/h, SuCost $}}
gen_data = {
    1: {'min': 40, 'max': 80, 'RR': 25, 'OpCost': 10, 'NlCost': 100, 'SuCost': 800},
    2: {'min': 20, 'max': 90, 'RR': 30, 'OpCost': 30, 'NlCost':  50, 'SuCost': 100},
}

# Periods: {t: total system load MW}
period_data = {1: 70, 2: 110}

In [3]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

# ── Sets ─────────────────────────────────────────────────────────
model.GEN    = Set(initialize=gen_data.keys(),    ordered=True)
model.PERIOD = Set(initialize=period_data.keys(), ordered=True)

# ── Parameters ───────────────────────────────────────────────────
model.gen_min    = Param(model.GEN,    initialize={g: gen_data[g]['min']    for g in gen_data})
model.gen_max    = Param(model.GEN,    initialize={g: gen_data[g]['max']    for g in gen_data})
model.gen_RR     = Param(model.GEN,    initialize={g: gen_data[g]['RR']     for g in gen_data})
model.gen_OpCost = Param(model.GEN,    initialize={g: gen_data[g]['OpCost'] for g in gen_data})
model.gen_NlCost = Param(model.GEN,    initialize={g: gen_data[g]['NlCost'] for g in gen_data})
model.gen_SuCost = Param(model.GEN,    initialize={g: gen_data[g]['SuCost'] for g in gen_data})
model.TotalPd    = Param(model.PERIOD, initialize=period_data)

# ── Decision Variables ────────────────────────────────────────────
model.u  = Var(model.GEN, model.PERIOD, within=Binary,           doc='Commitment status (1=on)')
model.v  = Var(model.GEN, model.PERIOD, within=Binary,           doc='Startup indicator (1=starting up)')
model.Pg = Var(model.GEN, model.PERIOD, within=NonNegativeReals, doc='Power output (MW)')

# ── Objective: minimise total cost ───────────────────────────────
def obj_rule(m):
    return sum(
        m.gen_OpCost[g] * m.Pg[g, t]
        + m.gen_NlCost[g] * m.u[g, t]
        + m.gen_SuCost[g] * m.v[g, t]
        for g in m.GEN for t in m.PERIOD
    )
model.obj = Objective(rule=obj_rule, sense=minimize)

# ── Power balance ─────────────────────────────────────────────────
def PowerBalance_rule(m, t):
    return sum(m.Pg[g, t] for g in m.GEN) == m.TotalPd[t]
model.PowerBalance = Constraint(model.PERIOD, rule=PowerBalance_rule)

# ── Generation limits: Pgmin·u ≤ Pg ≤ Pgmax·u ────────────────────
def genLimitMin_rule(m, g, t):
    return m.gen_min[g] * m.u[g, t] <= m.Pg[g, t]
model.genLimitMin = Constraint(model.GEN, model.PERIOD, rule=genLimitMin_rule)

def genLimitMax_rule(m, g, t):
    return m.Pg[g, t] <= m.gen_max[g] * m.u[g, t]
model.genLimitMax = Constraint(model.GEN, model.PERIOD, rule=genLimitMax_rule)

# ── Ramp-up limit (BigM relaxed during startup) ───────────────────
# Pg[g,t] - Pg[g,t-1] <= RR·u[g,t-1] + BigM·v[g,t]
def genRRUp_rule(m, g, t):
    if t == m.PERIOD.first(): return Constraint.Skip
    tp = m.PERIOD.prev(t)
    return m.Pg[g, t] - m.Pg[g, tp] <= m.gen_RR[g] * m.u[g, tp] + BigM * m.v[g, t]
model.genRRUp = Constraint(model.GEN, model.PERIOD, rule=genRRUp_rule)

# ── Ramp-down limit (BigM relaxed during shutdown) ────────────────
# Pg[g,t-1] - Pg[g,t] <= RR·u[g,t] + BigM·(v[g,t] - u[g,t] + u[g,t-1])
def genRRDn_rule(m, g, t):
    if t == m.PERIOD.first(): return Constraint.Skip
    tp = m.PERIOD.prev(t)
    return m.Pg[g, tp] - m.Pg[g, t] <= m.gen_RR[g] * m.u[g, t] + BigM * (m.v[g, t] - m.u[g, t] + m.u[g, tp])
model.genRRDn = Constraint(model.GEN, model.PERIOD, rule=genRRDn_rule)

# ── Startup logic: v[g,t] >= u[g,t] - u[g,t-1] ──────────────────
def genVU_rule(m, g, t):
    if t == m.PERIOD.first():
        return m.v[g, t] >= m.u[g, t]          # initial period: startup if committed
    tp = m.PERIOD.prev(t)
    return m.v[g, t] >= m.u[g, t] - m.u[g, tp]
model.genVU = Constraint(model.GEN, model.PERIOD, rule=genVU_rule)

In [4]:
# ─────────────────────────────────────────────────────────────────
# Solve
# ─────────────────────────────────────────────────────────────────
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

# ─────────────────────────────────────────────────────────────────
# Display Results
# ─────────────────────────────────────────────────────────────────
total_cost = sum(
    gen_data[g]['OpCost'] * model.Pg[g, t].value
    + gen_data[g]['NlCost'] * model.u[g, t].value
    + gen_data[g]['SuCost'] * model.v[g, t].value
    for g in model.GEN for t in model.PERIOD
)
print(f'\n=== Optimal Total Cost = ${total_cost:.2f} ===\n')
print(f'  {"Gen":<5} {"Period":<8} {"u":>4} {"v":>4} {"Pg (MW)":>10}')
print('  ' + '-' * 35)
for g in model.GEN:
    for t in model.PERIOD:
        print(f'  G{g:<4} t={t:<6} '
              f'{int(round(model.u[g,t].value)):>4} '
              f'{int(round(model.v[g,t].value)):>4} '
              f'{model.Pg[g,t].value:>10.2f}')
print(f'\nTotal solve elapsed time: {solve_time:.4f} seconds')

Set parameter Username
Set parameter LicenseID to value 2708162
Academic license - for non-commercial use only - expires 2026-09-12
Read LP format model from file /var/folders/zx/xyzw1dz90lv0dgc7dt781qf80000gn/T/tmpwom6cjk2.pyomo.lp
Reading time = 0.00 seconds
x1: 18 rows, 12 columns, 48 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.3.0 25D125)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  90
MIPGap  0

Optimize a model with 18 rows, 12 columns and 48 nonzeros
Model fingerprint: 0x6253182a
Variable types: 4 continuous, 8 integer (8 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+01, 8e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [7e+01, 1e+02]
Presolve removed 8 rows and 6 columns
Presolve time: 0.00s
Presolved: 10 rows, 6 columns, 30 nonzeros
V